<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/04_Quantization_Application_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Quantization Application 실습**

**실습 목표**
1. FP32 baseline 모델의 정확도, 크기, latency를 측정합니다.
2. Dynamic Quantization을 적용하고 결과를 비교합니다.
3. Static Quantization에서 activation observer와 calibration 과정을 확인합니다.
4. calibration batch 수가 결과에 미치는 영향을 확인합니다.
5. ResNet 모델에서 quantization 적용 시 고려할 점을 확인합니다.

# 0. Colab 환경설정
- colab에서 GPU를 사용할 수 있도록 세팅
    - 런타임 > 런타임 유형 변경 > Python 3 와 T4 GPU 선택
- colab에서 Google Drive에 접근할 수 있도록 설정

In [2]:
from google.colab import drive
drive.mount('/content/drive')

print("[현재 파일 위치]")
!pwd
print("[현재 디렉토리의 파일 확인]")
!ls

Mounted at /content/drive
[현재 파일 위치]
/content
[현재 디렉토리의 파일 확인]
drive  sample_data


**day 2** 폴더로 이동해주세요.

왼쪽의 **폴더** 아이콘을 클릭하면 경로를 쉽게 볼 수 있습니다.

In [3]:
# 본인 환경에 맞게 경로를 수정하여 사용하세요.
%cd /content/drive/MyDrive/day2
!pwd

/content/drive/.shortcut-targets-by-id/1-vYRZOLYLvUD5Kma-yELIfM0X7TGxy15/day2
/content/drive/.shortcut-targets-by-id/1-vYRZOLYLvUD5Kma-yELIfM0X7TGxy15/day2


# 1. 실험 준비

이번 실습에서는 CIFAR-10 기반의 ResNet-18 모델을 사용합니다.

- Dynamic Quantization: 주로 Linear, LSTM, GRU에 적용되며, CNN에서는 효과가 제한적일 수 있습니다.
- Static Quantization: weight와 activation을 모두 양자화하기 위해 calibration이 필요합니다.
- QAT: 학습 중 fake quantization을 삽입하여 양자화 오차를 미리 반영합니다.

수업 시간을 고려하여 평가 batch 수는 제한해서 사용합니다.

In [4]:
import os
import copy
import warnings

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models.quantization import resnet18
import torch.ao.quantization as quantization
from torch.ao.quantization import QConfigMapping
import torch.ao.quantization.quantize_fx as quantize_fx

from utils import *

warnings.filterwarnings("ignore")
set_random_seeds(1234)


## 1-1. Quantization Backend 설정

PyTorch quantization은 CPU backend에 따라 지원되는 연산과 성능이 달라질 수 있습니다.

- x86 CPU에서는 일반적으로 `fbgemm`을 사용합니다.
- ARM 계열 환경에서는 `qnnpack`을 사용하는 경우가 많습니다.

Colab CPU 환경에서는 보통 `fbgemm`이 사용 가능합니다.


In [5]:
# quantization backend 설정
if 'fbgemm' in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = 'fbgemm'
elif 'x86' in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = 'x86'
else:
    torch.backends.quantized.engine = torch.backends.quantized.supported_engines[0]

print(f"Quantization backend: {torch.backends.quantized.engine}")


Quantization backend: fbgemm


## 1-2. CIFAR-10 Dataset 준비

기존 `Quantization_full.ipynb`와 동일하게 CIFAR-10 데이터셋을 사용합니다.

`prepare_dataloader()` 함수는 `utils.py`에 정의되어 있으며, ResNet-18 입력 크기인 224×224에 맞게 CIFAR-10 이미지를 resize합니다.

실습 시간 단축을 위해 평가 batch 수는 제한해서 사용합니다. 더 정확한 결과가 필요하면 `EVAL_MAX_BATCHES`와 `BENCHMARK_BATCHES` 값을 늘리면 됩니다.


In [6]:
# 실습 시간 조절용 설정값
EVAL_MAX_BATCHES = 10
BENCHMARK_BATCHES = 10
CALIBRATION_BATCHES = 10

# CIFAR-10 dataloader 준비
trainloader, testloader = prepare_dataloader()

# device 설정
cuda_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
cpu_device = torch.device("cpu")

print("Train batches:", len(trainloader))
print("Test batches:", len(testloader))
print("CUDA device:", cuda_device)
print("CPU device:", cpu_device)


[CIFAR-10] Found local archive: ./data/cifar-10-python.tar.gz
[CIFAR-10] Extracting to: /content/data
Train batches: 391
Test batches: 313
CUDA device: cuda:0
CPU device: cpu


## 1-3. ResNet-18 모델 생성 함수

CIFAR-10은 class 수가 10개이므로 ResNet-18의 마지막 fully connected layer를 10개 출력으로 바꿉니다.

이 실습에서는 `resnet18_cifar10.pt` checkpoint를 사용하므로 ImageNet pretrained weight를 새로 다운로드하지 않습니다. 즉, 모델 구조를 만든 뒤 CIFAR-10 checkpoint를 로드합니다.


In [7]:
def build_resnet18_cifar10():
    # Quantization 가능한 torchvision ResNet-18 구조를 생성합니다.
    # checkpoint를 불러올 것이므로 weights=None을 사용합니다.
    ###### [실습] `resnet18` 모델을 불러오고, quantization 적용 전 FP32 모델 구조를 생성하세요. ######
    model = resnet18(weights=None, quantize=False)

    ###### [실습] CIFAR-10 클래스 수에 맞게 최종 레이어(fc)를 10개 출력으로 수정하세요. ######
    model.fc = nn.Linear(model.fc.in_features,10)
    return model

model_fp32 = build_resnet18_cifar10()
print(model_fp32.__class__)
print(model_fp32.fc)


<class 'torchvision.models.quantization.resnet.QuantizableResNet'>
Linear(in_features=512, out_features=10, bias=True)


# 2. FP32 Baseline 모델 로드 및 평가

이 단계에서는 사전 학습된 `resnet18_cifar10.pt`를 불러옵니다.

수업 시간 안에 ResNet-18을 CIFAR-10에 처음부터 학습하면 시간이 오래 걸리므로, 아래의 학습 코드는 참고용으로만 남겨두고 기본 실행에서는 checkpoint를 로드합니다.

```python
train_model(model_fp32, trainloader, testloader, criterion, optimizer, scheduler, cuda_device, epochs=10)
save_model(model=model_fp32, model_dir=model_dir, model_filename=model_filename)
```


In [8]:
# 학습 설정: checkpoint가 없을 때 참고용으로 사용할 수 있습니다.
###### [실습] 분류 손실 함수(nn.CrossEntropyLoss), SGD 옵티마이저(optim.SGD) 및 MultiStepLR 학습률 스케줄러를 설정해보세요. ######
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_fp32.parameters(), lr=1e-1, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1, last_epoch=-1)

# 모델 저장 경로 설정
model_dir = "saved_models"
model_filename = "resnet18_cifar10.pt"
model_filepath = os.path.join(model_dir, model_filename)

# 기본 모델 학습: 수업 시간상 기본 실행에서는 사용하지 않습니다.
# train_model(model_fp32, trainloader, testloader, criterion, optimizer, scheduler, cuda_device, epochs=10)
# save_model(model=model_fp32, model_dir=model_dir, model_filename=model_filename)

if not os.path.exists(model_filepath):
    raise FileNotFoundError(
        f"{model_filepath} 파일이 없습니다. "
        "saved_models 폴더에 resnet18_cifar10.pt를 넣은 뒤 다시 실행하세요."
    )

# 학습된 모델 불러오기
model_fp32 = load_model(model=model_fp32, model_filepath=model_filepath, device=cuda_device)
model_fp32.eval()

# 모델 정확도 평가
fp32_accuracy = evaluate_model(model_fp32, testloader, cpu_device, max_batches=EVAL_MAX_BATCHES)

# 모델 크기 측정
fp32_model_size = get_size_of_model(model_fp32)

# 모델 지연 시간 및 추론 시간 평가
fp32_latency, fp32_inference_time = benchmark_model(model_fp32, testloader, cpu_device, num_batches=BENCHMARK_BATCHES)

# 결과 출력
print('[Baseline FP32 ResNet-18]')
print(f"Accuracy: {fp32_accuracy:.2f}%")
print(f"Model Size: {(fp32_model_size/1e3):.2f} MB")
print(f"Average Latency: {fp32_latency:.6f} seconds")
print(f"Inference Time per Image: {fp32_inference_time:.6f} seconds")


[Baseline FP32 ResNet-18]
Accuracy: 88.12%
Model Size: 44.80 MB
Average Latency: 1.736170 seconds
Inference Time per Image: 0.054255 seconds


# 3. Activation Range와 Calibration 확인

Static quantization에서는 weight뿐 아니라 activation도 quantization 대상이 됩니다.

Activation은 입력 데이터에 따라 값의 범위가 달라지므로, calibration data를 모델에 통과시켜 각 layer의 activation range를 관찰해야 합니다.

아래에서는 먼저 hook 기반으로 FP32 모델의 주요 activation range를 수집해봅니다. 이 결과는 실제 PyTorch observer와 완전히 동일한 방식은 아니지만, calibration이 왜 필요한지 이해하는 데 도움이 됩니다.


In [9]:
# FP32 모델의 activation range를 hook으로 수집합니다.
activation_ranges = collect_activation_ranges(
    model=model_fp32,
    dataloader=trainloader,
    device=cpu_device,
    num_batches=3,
)

print_activation_ranges(activation_ranges, max_items=12)


[Activation range]
conv1                          | min: -44.215088 | max:  55.022717
relu                           | min:  0.000000 | max:  12.186639
layer1.0.conv1                 | min: -34.114655 | max:  24.529850
layer1.0.relu                  | min:  0.000000 | max:  6.115302
layer1.0.conv2                 | min: -12.492929 | max:  8.767732
layer1.1.conv1                 | min: -34.367958 | max:  27.374704
layer1.1.relu                  | min:  0.000000 | max:  7.402863
layer1.1.conv2                 | min: -15.769772 | max:  12.655544
layer2.0.conv1                 | min: -32.167446 | max:  28.684221
layer2.0.relu                  | min:  0.000000 | max:  8.613732
layer2.0.conv2                 | min: -21.938423 | max:  18.604027
layer2.0.downsample.0          | min: -8.780705 | max:  11.542987


# 4. Post-Training Dynamic Quantization

Dynamic Quantization은 모델 학습이 끝난 뒤 적용하는 방식입니다.

특징은 다음과 같습니다.

1. weight는 미리 quantization합니다.
2. activation은 inference 시점에 동적으로 quantization합니다.
3. 일반적으로 `nn.Linear`, RNN 계열 layer에 적용하기 쉽습니다.
4. CNN에서는 convolution layer가 그대로 남는 경우가 많으므로, 효과가 제한적일 수 있습니다.

여기서는 기존 `Quantization_full.ipynb`와 같이 ResNet-18의 `fc` layer를 중심으로 dynamic quantization을 적용합니다.

ResNet-18은 대부분 convolution layer로 구성되어 있기 때문에, 이 실습의 Dynamic Quantization에서는 주로 마지막 `fc` layer에만 quantization 효과가 적용됩니다.

따라서 model size 감소가 Static Quantization보다 작게 나타나는 것이 정상입니다.

In [10]:
# 예제 입력 생성: FX graph mode quantization에서 모델 trace에 사용합니다.
image, label = next(iter(trainloader))
example_inputs = (image,)

# 모델 복사
model_PTD = copy.deepcopy(model_fp32).to(cpu_device)
model_PTD.eval()

# qconfig 설정: fc 레이어에 dynamic quantization 적용
qconfig_mapping = QConfigMapping().set_module_name("fc", quantization.default_dynamic_qconfig)

# FX graph mode에서 prepare 및 convert 적용
model_PTD_prepared = quantize_fx.prepare_fx(model_PTD, qconfig_mapping, example_inputs)
model_PTD = quantize_fx.convert_fx(model_PTD_prepared)

# 평가를 위해 CPU로 이동
model_PTD.eval()
model_PTD = model_PTD.to(cpu_device)

# 정확도, 모델 크기, 지연 시간 평가
PTD_accuracy = evaluate_model(model_PTD, testloader, cpu_device, max_batches=EVAL_MAX_BATCHES)
PTD_model_size = get_size_of_model(model_PTD)
PTD_latency, PTD_inference_time = benchmark_model(model_PTD, testloader, cpu_device, num_batches=BENCHMARK_BATCHES)

# 결과 출력
print('[Post Quantization Dynamic]')
print(f"Accuracy: {PTD_accuracy:.2f}%")
print(f"Model Size: {(PTD_model_size/1e3):.2f} MB")
print(f"Average Latency: {PTD_latency:.6f} seconds")
print(f"Inference Time per Image: {PTD_inference_time:.6f} seconds")


[Post Quantization Dynamic]
Accuracy: 88.12%
Model Size: 44.71 MB
Average Latency: 1.784574 seconds
Inference Time per Image: 0.055768 seconds


# 5. Post-Training Static Quantization (PTQ)

Static Quantization은 모델 학습이 끝난 뒤 weight와 activation을 모두 quantization하는 방식입니다.

Dynamic Quantization과의 가장 큰 차이는 activation range를 미리 추정한다는 점입니다. 이를 위해 calibration 단계가 필요합니다.

실제 과정은 다음과 같습니다.

1. FP32 모델 복사
2. Layer fusion 적용
3. qconfig 설정
4. prepare 단계에서 observer 삽입
5. calibration data를 모델에 통과시켜 activation range 수집
6. convert 단계에서 quantized model 생성
7. accuracy, model size, latency 평가


## 5-1. Observer 삽입 및 Calibration 확인

`prepare`를 적용하면 모델 내부에 observer가 삽입됩니다.

Observer는 calibration data가 통과할 때 activation의 min/max를 관찰하고, 이를 바탕으로 scale과 zero-point를 계산합니다.


In [11]:
# PTQ를 위한 모델 복사
model_PTS_prepared = copy.deepcopy(model_fp32).to(cpu_device)
model_PTS_prepared.eval()

# 레이어 퓨전 적용
model_PTS_prepared.fuse_model()

# qconfig 설정
###### [실습] `quantization.get_default_qconfig`를 사용하여 현재 quantization backend에 맞는 기본 qconfig를 설정하세요. ######
model_PTS_prepared.qconfig = quantization.get_default_qconfig(torch.backends.quantized.engine)

# prepare 단계: observer 삽입
###### [실습] `quantization.prepare`를 사용하여 모델 내부에 observer를 삽입하세요. ######
model_PTS_prepared = quantization.prepare(model_PTS_prepared, inplace=False)

# calibration 전 observer 확인
print("[Calibration 전]")
print_observer_ranges(model_PTS_prepared, max_items=10)

# calibration 수행
calibrate_model(model_PTS_prepared, trainloader, cpu_device, num_batches=CALIBRATION_BATCHES)

# calibration 후 observer 확인
print("\n[Calibration 후]")
print_observer_ranges(model_PTS_prepared, max_items=10)


[Calibration 전]
[Observer activation range]
conv1                          | min: inf | max: -inf
layer1.0.conv1                 | min: inf | max: -inf
layer1.0.conv2                 | min: inf | max: -inf
layer1.0.add_relu              | min: inf | max: -inf
layer1.1.conv1                 | min: inf | max: -inf
layer1.1.conv2                 | min: inf | max: -inf
layer1.1.add_relu              | min: inf | max: -inf
layer2.0.conv1                 | min: inf | max: -inf
layer2.0.conv2                 | min: inf | max: -inf
layer2.0.downsample.0          | min: inf | max: -inf

[Calibration 후]
[Observer activation range]
conv1                          | min: 0.0 | max: 11.978970527648926
layer1.0.conv1                 | min: 0.0 | max: 6.4369425773620605
layer1.0.conv2                 | min: -3.6676783561706543 | max: 4.1230082511901855
layer1.0.add_relu              | min: 0.0 | max: 11.852291107177734
layer1.1.conv1                 | min: 0.0 | max: 7.6649065017700195
layer1.1.conv2 

## 5-2. Static Quantization 변환 및 평가

Calibration이 끝난 모델은 `convert`를 통해 실제 quantized module로 변환할 수 있습니다.

이때 convolution, linear 등의 일부 연산이 quantized operator로 바뀝니다.


In [12]:
# 양자화된 모델로 변환
###### [실습] `quantization.convert` 함수를 사용해 calibration이 끝난 모델을 최종 quantized model로 변환하세요. ######
model_PTS = quantization.convert(model_PTS_prepared, inplace=False)

# CPU에서 평가 준비
model_PTS.eval()
model_PTS = model_PTS.to(cpu_device)

# 정확도, 모델 크기, 지연 시간 평가
PTS_accuracy = evaluate_model(model_PTS, testloader, cpu_device, max_batches=EVAL_MAX_BATCHES)
PTS_model_size = get_size_of_model(model_PTS)
PTS_latency, PTS_inference_time = benchmark_model(model_PTS, testloader, cpu_device, num_batches=BENCHMARK_BATCHES)

# 결과 출력
print('[Post Quantization Static]')
print(f"Accuracy: {PTS_accuracy:.2f}%")
print(f"Model Size: {(PTS_model_size/1e3):.2f} MB")
print(f"Average Latency: {PTS_latency:.6f} seconds")
print(f"Inference Time per Image: {PTS_inference_time:.6f} seconds")


[Post Quantization Static]
Accuracy: 88.12%
Model Size: 11.31 MB
Average Latency: 1.000378 seconds
Inference Time per Image: 0.031262 seconds


## 5-3. Calibration Batch 수 변경 실험

Calibration data가 너무 적으면 activation range를 제대로 추정하지 못할 수 있습니다.

아래에서는 calibration batch 수를 바꿔가며 Static Quantization 결과가 어떻게 달라지는지 확인합니다.


In [13]:
def run_static_quantization_resnet(model_fp32, calibration_batches):
    # PTQ를 위한 모델 복사
    model_static = copy.deepcopy(model_fp32).to(cpu_device)
    model_static.eval()

    # 레이어 퓨전 적용
    model_static.fuse_model()

    # qconfig 설정
    model_static.qconfig = quantization.get_default_qconfig(torch.backends.quantized.engine)

    # prepare 단계: observer 삽입
    model_prepared = quantization.prepare(model_static, inplace=False)

    # calibration 단계: 일부 train batch를 통과시켜 activation range 관찰
    calibrate_model(model_prepared, trainloader, cpu_device, num_batches=calibration_batches)

    # convert 단계: observer 정보를 바탕으로 quantized model로 변환
    model_int8 = quantization.convert(model_prepared, inplace=False)
    model_int8.eval()

    # evaluation 단계: accuracy, model size, latency 측정
    acc = evaluate_model(model_int8, testloader, cpu_device, max_batches=EVAL_MAX_BATCHES)
    size = get_size_of_model(model_int8)
    latency, inference_time = benchmark_model(model_int8, testloader, cpu_device, num_batches=BENCHMARK_BATCHES)

    return model_int8, acc, size, latency, inference_time

calibration_results = []

###### [실습] calibration batch 수를 바꿔가며 accuracy, model size, latency가 어떻게 달라지는지 확인하세요. ######
for n_batches in [1, 5, 10, 20]:
    _, acc, size, latency, infer_time = run_static_quantization_resnet(model_fp32, calibration_batches=n_batches)
    calibration_results.append((n_batches, acc, size, latency, infer_time))
    print(f"Calibration Batches: {n_batches:2d} | Accuracy: {acc:.2f}% | Size: {size/1e3:.2f} MB | Latency: {latency:.6f} sec")


Calibration Batches:  1 | Accuracy: 87.50% | Size: 11.31 MB | Latency: 0.985732 sec
Calibration Batches:  5 | Accuracy: 88.44% | Size: 11.31 MB | Latency: 0.990155 sec
Calibration Batches: 10 | Accuracy: 88.12% | Size: 11.31 MB | Latency: 1.005229 sec
Calibration Batches: 20 | Accuracy: 88.12% | Size: 11.31 MB | Latency: 1.087935 sec


KeyboardInterrupt: 

# 6. Quantization Aware Training (QAT)

QAT는 학습 중에 fake quantization을 삽입하여 quantization으로 인해 발생할 오차를 미리 반영하는 방식입니다.

- 실제 INT8 연산을 바로 수행하는 것은 아닙니다.
- 학습 중에는 FP32 weight를 유지하면서 quantization 효과를 시뮬레이션합니다.
- 학습이 끝난 뒤 `convert`를 통해 실제 quantized model로 변환합니다.

기존 `Quantization_full.ipynb`와 동일하게 `resnet18_layerfusion_cifar10.pt` checkpoint를 사용합니다.


In [14]:
# Quantization Aware Training (QAT)를 위한 모델 복사 및 설정
model_QAT = copy.deepcopy(model_fp32).to(cpu_device)

# 모델의 레이어 퓨전(fusion) 적용
###### [실습] 모델에 레이어 퓨전(`fuse_model`)을 적용하여 QAT에 적합한 구조로 준비하세요. ######
model_QAT.fuse_model()

# QAT용 qconfig 설정
model_QAT.qconfig = quantization.get_default_qat_qconfig(torch.backends.quantized.engine)

# QAT 준비
model_QAT.train()
quantization.prepare_qat(model_QAT, inplace=True)

# 모델 저장 경로 설정
quantized_model_filename = "resnet18_layerfusion_cifar10.pt"
quantized_model_filepath = os.path.join(model_dir, quantized_model_filename)

# 손실 함수, 옵티마이저 및 학습률 스케줄러 설정: checkpoint가 없을 때 참고용입니다.
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_QAT.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

# QAT 모델 학습 및 저장: 수업 시간상 기본 실행에서는 사용하지 않습니다.
# train_model(model_QAT, trainloader, testloader, criterion, optimizer, scheduler, cpu_device, epochs=1)
# save_model(model=model_QAT, model_dir=model_dir, model_filename=quantized_model_filename)

if not os.path.exists(quantized_model_filepath):
    raise FileNotFoundError(
        f"{quantized_model_filepath} 파일이 없습니다. "
        "saved_models 폴더에 resnet18_layerfusion_cifar10.pt를 넣은 뒤 다시 실행하세요."
    )

# 학습된 QAT 모델 불러오기
model_QAT = load_model(model=model_QAT, model_filepath=quantized_model_filepath, device=cpu_device)

# 양자화된 모델로 변환
###### [실습] `quantization.convert` 함수를 사용해 QAT가 적용된 모델을 최종 quantized model로 변환하세요. ######
model_QAT = quantization.convert(model_QAT.eval(), inplace=False)

# CPU에서 양자화된 모델 평가
model_QAT.eval()
model_QAT = model_QAT.to(cpu_device)

# 정확도, 모델 크기, 지연 시간 평가
QAT_accuracy = evaluate_model(model_QAT, testloader, cpu_device, max_batches=EVAL_MAX_BATCHES)
QAT_model_size = get_size_of_model(model_QAT)
QAT_latency, QAT_inference_time = benchmark_model(model_QAT, testloader, cpu_device, num_batches=BENCHMARK_BATCHES)

# 결과 출력
print('[Quantization Aware Training]')
print(f"Accuracy: {QAT_accuracy:.2f}%")
print(f"Model Size: {(QAT_model_size/1e3):.2f} MB")
print(f"Average Latency: {QAT_latency:.6f} seconds")
print(f"Inference Time per Image: {QAT_inference_time:.6f} seconds")


[Quantization Aware Training]
Accuracy: 90.62%
Model Size: 11.31 MB
Average Latency: 1.042410 seconds
Inference Time per Image: 0.032575 seconds


# 7. 결과 분석

아래 표는 ResNet-18 CIFAR-10 모델에 대해 FP32 baseline, Dynamic Quantization, Static Quantization, QAT 결과를 비교합니다.

실제 결과는 실행 환경, CPU backend, calibration batch 수, checkpoint 성능에 따라 달라질 수 있습니다.


In [15]:
print("Model: ResNet-18 CIFAR-10".ljust(50), "Accuracy (%)".center(15), "Model Size (MB)".center(20), "Avg Latency (sec)".center(20), "Inference Time (sec)".center(22))
print("=" * 132)

print("Baseline FP32".ljust(50),
      f"{fp32_accuracy:.2f}".center(15),
      f"{fp32_model_size/1e3:.2f}".center(20),
      f"{fp32_latency:.6f}".center(20),
      f"{fp32_inference_time:.6f}".center(22))

print("Post Quantization Dynamic".ljust(50),
      f"{PTD_accuracy:.2f}".center(15),
      f"{PTD_model_size/1e3:.2f}".center(20),
      f"{PTD_latency:.6f}".center(20),
      f"{PTD_inference_time:.6f}".center(22))

print("Post Quantization Static".ljust(50),
      f"{PTS_accuracy:.2f}".center(15),
      f"{PTS_model_size/1e3:.2f}".center(20),
      f"{PTS_latency:.6f}".center(20),
      f"{PTS_inference_time:.6f}".center(22))

print("Quantization Aware Training".ljust(50),
      f"{QAT_accuracy:.2f}".center(15),
      f"{QAT_model_size/1e3:.2f}".center(20),
      f"{QAT_latency:.6f}".center(20),
      f"{QAT_inference_time:.6f}".center(22))


Model: ResNet-18 CIFAR-10                            Accuracy (%)    Model Size (MB)     Avg Latency (sec)    Inference Time (sec) 
Baseline FP32                                           88.12             44.80               1.736170              0.054255       
Post Quantization Dynamic                               88.12             44.71               1.784574              0.055768       
Post Quantization Static                                88.12             11.31               1.000378              0.031262       
Quantization Aware Training                             90.62             11.31               1.042410              0.032575       


### 결과 해석 시 주의할 점

위 결과는 실습 시간을 줄이기 위해 CIFAR-10 test set 전체가 아니라 일부 batch만 사용하여 측정한 값입니다.

따라서 accuracy와 latency 수치는 최종 benchmark 값이라기보다는 quantization 방식 간의 경향을 비교하기 위한 실습용 결과로 해석해야 합니다.

특히 평가 batch 수가 제한되어 있기 때문에 QAT 결과가 FP32 baseline보다 높게 나올 수도 있습니다.

중요한 점은 QAT가 학습 중 quantization 효과를 미리 반영하여, PTQ 대비 정확도 손실을 줄이는 데 유리할 수 있다는 것입니다.

## 7-1. 결론 및 추천

1. 정확도 우선
    - QAT는 학습 과정에서 quantization 효과를 반영하므로 정확도 손실을 줄이는 데 유리합니다.
    - 다만 추가 학습 비용이 필요합니다.

2. 메모리 제약이 큰 경우
    - Static Quantization은 weight와 activation을 모두 INT8로 변환할 수 있어 모델 크기와 latency 개선에 유리합니다.
    - calibration data가 실제 입력 분포를 잘 대표해야 합니다.

3. 간단한 적용이 필요한 경우
    - Dynamic Quantization은 코드 수정이 적고 적용이 간단합니다.
    - 하지만 CNN에서는 convolution layer가 그대로 남을 수 있어 효과가 제한적일 수 있습니다.

4. 실제 제품 적용 관점
    - Quantization 성능은 backend, operator support, hardware에 영향을 받습니다.
    - 따라서 accuracy뿐 아니라 model size, latency, memory를 함께 측정해야 합니다.